# Evaluate Models on MIMIC-IV and MIMIC-III

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pytorch_lightning as pl
from matplotlib import pyplot as plt

In [ ]:
import lab_ecg
from lab_ecg import classification, config, dataset

In [ ]:
# specify the weights.ckpt of the model to evaluate
weights = Path("checkpoints/epoch=19-step=3220-val_loss=0.54.ckpt")

params = classification.load_params(weights.parent / "params.yaml")

### Data

Evaluate either on MIMIC-IV or MIMIC-III. Note that only single-lead models can be evaluated on MIMIC-III.

#### MIMIC-IV

In [ ]:
store = dataset.HDFSignalDatastore(
    "data/mimic4_samples.h5",
    "data/mimic4_signals.h5",
    src_channels=["I", "II", "III", "aVR", "aVF", "aVL", "V1", "V2", "V3", "V4", "V5", "V6"],
    channels=params["data_store"]["channels"],
    label_cols=params["data_store"]["label_cols"][:1],
    binary_thresholds=params["data_store"]["binary_thresholds"],
    fs=500,  # sample rate of source signals => do not change!
    memory_limit="40GB",
)
store.feature_cols = ["subject_id", "sample_id", "male", "age"]
len(store)

#### MIMIC-III

In [ ]:
assert len(params["data_store"]["channels"]) == 1, "Only single-lead models can be evaluated on MIMIC-III"

In [ ]:
store = dataset.HDFSignalDatastore(
    "data/mimic3_samples.h5",
    "data/mimic3_signals.h5",
    src_channels=["II"],
    channels=["II"],
    label_cols=params["data_store"]["label_cols"][:1],
    binary_thresholds=params["data_store"]["binary_thresholds"],
    fs=125,  # sample rate of source signals => do not change!
    memory_limit="40GB",
)
store.feature_cols = ["subject_id", "sample_id", "male", "age"]
len(store)

### Config

In [ ]:
transform, test_transform = config.build_pipeline(params, store)

### Data Loader

In [ ]:
split = params["data_store"]["label_cols"][0] + "_split"
test_indices = np.flatnonzero(store.df[split] == "test")  # you can evaluate on a different subset, too
len(test_indices)

In [ ]:
test_loader = dataset.SignalDataset(store, indices=test_indices, transform=test_transform, auto_open=True).dataloader(
    batch_size=params["training"]["batch_size"], num_workers=0 if store.in_memory else 4
)
print(len(test_loader.dataset))

In [ ]:
model = classification.make(params, weights=weights).eval().cuda()

### Apply Model

In [ ]:
predictor = pl.Trainer(logger=False, devices=1, callbacks=[])

In [ ]:
pred = predictor.predict(model, dataloaders=test_loader)
test_loader.dataset.close()

In [ ]:
# a single sample could be split into multiple snippets, whose predictions need to be averaged to obtain the final
# prediction
sample_ids = np.concatenate([np.asanyarray(p[test_loader.dataset.store.idx_name]) for p in pred])
sample_ids.shape

In [ ]:
# ground-truth
y_binary = (
    pd.DataFrame(
        data=np.concatenate([p["y_binary"].numpy() for p in pred]),
        index=sample_ids,
        columns=test_loader.dataset.store.binary_labels,
    )
    .groupby(level=0)
    .first()
)
y_binary.index.name = test_loader.dataset.store.idx_name

In [ ]:
# predictions
y_binary_hat = (
    pd.DataFrame(
        data=np.concatenate([p["y_binary_hat"].numpy() for p in pred]),
        index=sample_ids,
        columns=test_loader.dataset.store.binary_labels,
    )
    .groupby(level=0)
    .mean()
)  # average predictions of each sample; you can choose a different aggregation, like max
y_binary_hat.index.name = test_loader.dataset.store.idx_name

### Evaluate

In [ ]:
# decision threshold for metrics like sensitivity, specificity, etc.
# should be determined on validation set
decision_threshold = 0.0

In [ ]:
# estimate bootstrapped model performance
bs = lab_ecg.evaluate(y_binary.iloc[:, 0], y_binary_hat.iloc[:, 0], y_binary_hat.iloc[:, 0] > decision_threshold)
avg = bs.agg("mean")
ci95_lo, ci95_hi = bs.agg("ci95")
pd.DataFrame([ci95_lo, avg, ci95_hi], index=["95CI_lo", "average", "95CI_hi"]).T

### Saliency Maps

In [ ]:
from lab_ecg import gradient

In [ ]:
# explain all samples in the first batch
batch = next(iter(test_loader))
args, m = model.get_args(batch)  # `m` is number of snippets per sample ("multiplicity")
with gradient.Gradient(model.network) as grd_model:
    out, expl = grd_model(*args)
expl = expl[0]  # waveform is first item of list; other items are age and gender
expl.shape

In [ ]:
# visualize
i = 0
print(
    "i =", (i // m, i % m), "| GT =", batch["binary_labels"][i // m].numpy(), "| P =", out[0][i].cpu().detach().numpy()
)
gradient.plot_saliency_map(
    args[0][i].numpy(),
    expl[i],
    channels=store.channels,
    figsize=(4, 8),
    yticks=False,
    channel_order=[6, 7, 8, 9, 10, 11, 1, 3, 2, 4, 0, 5],
)
plt.tight_layout()
plt.show();